# 日常回测控制台

这份 notebook 用于**日常使用**，目标是尽量少切换 cell，直接完成：

1. 读取本地市场数据  
2. 设置资产池与回测参数  
3. 运行风险平价回测  
4. 查看关键结果  
5. 导出回测结果

> 依赖：`market_data.py`、`backtest.py`、`risk_parity.py`


In [ ]:

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from market_data import create_manager, today_str, load_tushare_token
from backtest import build_market_matrices, simulate_risk_parity_backtest, calc_drawdown

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True

# ========= 路径与连接 =========
TUSHARE_TOKEN = load_tushare_token()
DB_PATH = "data/db/market_data.db"
EXPORT_DIR = Path("data/exports_risk_parity")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20150101",
    default_exchange="SSE",
)

print("manager ready")
print("db_path =", Path(DB_PATH).resolve())
print("export_dir =", EXPORT_DIR.resolve())


In [ ]:

# ========= 资产池 =========

#以下为无杠杆全天候国内260324版本
WATCHLIST = [
    "510300.SH",  # 沪深300ETF (核心权益：大盘蓝筹)
    #"510500.SH",  # 中证500ETF (核心权益：中盘成长)
    #"510170.SH",  # 50ETF (核心权益：大宗商品股票集合)
    #"159915.SZ",  # 创业板ETF (核心权益：高新成长 - 纯被动)
    "511090.SH",  # 30年国债ETF (避险资产：超长债，对利率极度敏感)
    #"511010.SH",  # 国债ETF (基础配置：中长期债)
    "518880.SH",  # 黄金ETF (抗通胀/避险：金属商品)
    "159981.SZ",  # 能源ETF (抗通胀：能源/电力)
    "159985.SZ",  # 豆粕ETF (抗通胀：农产品期货)
    "501018.SH",  # 南方原油LOF (抗通胀：国际原油价格)
    "515100.SH",  # 红利低波100ETF (抗通胀：红利股)
    #"513000.SH",  # 225日经ETF
    "159659.SZ",  # 100纳斯达克ETF
]
# ========= 数据区间 =========
START_DATE = "20230101"
END_DATE = today_str()

# ========= 回测参数 =========
BACKTEST_PARAMS = {
    "initial_cash": 1_000_000.0,
    #"initial_cash": 30_000.0,
    "lookback_window": 120,
    "rebalance_freq": "M",          # D / W / M / Q / Y
    "execution_price_type": "avg",  # open / close / high / low / avg
    "fee_rate_buy": 0.0005,
    "fee_rate_sell": 0.0005,
    "lot_size": 100,
    "max_trade_amount_ratio": 0.05,
    "amount_unit_scale": 1000.0,# 按照tushare文档，单位似乎是千元？
    "use_drift_trigger": False,
    "drift_threshold": 0.05,
    "risk_free_rate": 0.0,#设置为0，计算的夏普率就是“收益风险比”
    "annualization": 252,
}

RP_PREPARE_KWARGS = {
    "ffill": True,
    "ffill_limit": 5,
    "min_non_na_ratio": 0.8,
    "drop_all_na_dates": True,

}
valuation_ffill_limit = 5
RP_WEIGHT_KWARGS = {
    "method": "sample",      # sample / ewma
    "return_type": "log",    # log / simple
    "annualization": 252,
    "long_only": True,
    "drop_any_na": True,
}

print("watchlist =", WATCHLIST)
print("params loaded")


In [ ]:
recent_summary = manager.refresh_recent_daily_prices(
    ts_codes=WATCHLIST,
    lookback_days=180,#如果超过180天没用过，请先更新数据库
    end_date=today_str(),
)

print("最近数据刷新结果：")
print(recent_summary)

In [ ]:


# ========= 读取市场数据并构建 market 字典 =========
raw_prices = manager.store.get_daily_prices(
    ts_codes=WATCHLIST,
    start_date=START_DATE,
    end_date=END_DATE,
)

print("raw_prices shape =", raw_prices.shape)
display(raw_prices.head())

coverage = raw_prices.groupby("ts_code")["trade_date"].agg(["min", "max", "count"])
display(coverage)

market = build_market_matrices(
    data=raw_prices,
    fields=("open", "high", "low", "close", "amount"),
    date_col="trade_date",
    code_col="ts_code",
    date_format="%Y%m%d",
)

RP_PREPARE_KWARGS["calendar"] = market["close"].index

for k, v in market.items():
    print(k, v.shape)


In [ ]:

# ========= 运行回测 =========
result = simulate_risk_parity_backtest(
    market=market,
    initial_cash=BACKTEST_PARAMS["initial_cash"],
    lookback_window=BACKTEST_PARAMS["lookback_window"],
    rebalance_freq=BACKTEST_PARAMS["rebalance_freq"],
    execution_price_type=BACKTEST_PARAMS["execution_price_type"],
    fee_rate_buy=BACKTEST_PARAMS["fee_rate_buy"],
    fee_rate_sell=BACKTEST_PARAMS["fee_rate_sell"],
    lot_size=BACKTEST_PARAMS["lot_size"],
    max_trade_amount_ratio=BACKTEST_PARAMS["max_trade_amount_ratio"],
    amount_unit_scale=BACKTEST_PARAMS["amount_unit_scale"],
    use_drift_trigger=BACKTEST_PARAMS["use_drift_trigger"],
    drift_threshold=BACKTEST_PARAMS["drift_threshold"],
    rp_prepare_kwargs=RP_PREPARE_KWARGS,
    rp_weight_kwargs=RP_WEIGHT_KWARGS,
    risk_free_rate=BACKTEST_PARAMS["risk_free_rate"],
    annualization=BACKTEST_PARAMS["annualization"],
    valuation_ffill_limit=valuation_ffill_limit,
)

summary = result["summary"]
nav_df = result["nav_df"]
returns = result["returns"]
weights_df = result["weights_df"]
positions_df = result["positions_df"]
target_weights_df = result["target_weights_df"]
trades_df = result["trades_df"]
rebalance_log_df = result["rebalance_log_df"]
asset_corr_matrix = result["asset_corr_matrix"]
risk_contribution_df = result["risk_contribution_df"]

print("回测完成")
display(summary.to_frame("value"))


In [ ]:
# ========= 日常查看：净值 / 回撤 / 现金占比 / 最近调仓 / 最近交易 =========
drawdown = calc_drawdown(nav_df["nav"])
cash_weight = nav_df["cash"] / nav_df["nav"]

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# 1. 净值
axes[0].plot(nav_df.index, nav_df["nav"], label="NAV")
axes[0].set_title("Portfolio NAV")
axes[0].legend()

# 2. 回撤
axes[1].plot(drawdown.index, drawdown, label="Drawdown")
axes[1].set_title("Drawdown")
axes[1].legend()

# 3. 现金占比
axes[2].plot(cash_weight.index, cash_weight, label="Cash Weight")
axes[2].set_title("Cash Weight")
axes[2].legend()

plt.tight_layout()
plt.show()

print("最近净值：")
display(nav_df.tail())

print("最近现金占比：")
display(cash_weight.tail().to_frame("cash_weight"))

print("最近目标权重：")
display(target_weights_df.tail())

print("最近实际权重：")
display(weights_df.tail())

print("最近持仓：")
display(positions_df.tail())

print("最近调仓日志：")
display(rebalance_log_df.tail(10))

print("最近交易记录：")
display(trades_df.tail(20))

print("平均现金占比：", cash_weight.mean())
print("最大现金占比：", cash_weight.max())

In [ ]:

# ========= 一键导出 =========
nav_df.to_csv(EXPORT_DIR / "nav.csv")
weights_df.to_csv(EXPORT_DIR / "weights.csv")
positions_df.to_csv(EXPORT_DIR / "positions.csv")
target_weights_df.to_csv(EXPORT_DIR / "target_weights.csv")
trades_df.to_csv(EXPORT_DIR / "trades.csv", index=False)
rebalance_log_df.to_csv(EXPORT_DIR / "rebalance_log.csv", index=False)
summary.to_csv(EXPORT_DIR / "summary.csv")

asset_corr_matrix.to_csv(EXPORT_DIR / "asset_corr_matrix.csv")
risk_contribution_df.to_csv(EXPORT_DIR / "risk_contribution_df.csv")

print("export done")


In [ ]:

# ========= 可选：快速改参数重跑 =========
# 这里只留一个最常用示例，避免 notebook 过长

# BACKTEST_PARAMS["rebalance_freq"] = "M"
# BACKTEST_PARAMS["use_drift_trigger"] = True
# BACKTEST_PARAMS["drift_threshold"] = 0.08
# RP_WEIGHT_KWARGS["method"] = "ewma"

# result = simulate_risk_parity_backtest(
#     market=market,
#     initial_cash=BACKTEST_PARAMS["initial_cash"],
#     lookback_window=BACKTEST_PARAMS["lookback_window"],
#     rebalance_freq=BACKTEST_PARAMS["rebalance_freq"],
#     execution_price_type=BACKTEST_PARAMS["execution_price_type"],
#     fee_rate_buy=BACKTEST_PARAMS["fee_rate_buy"],
#     fee_rate_sell=BACKTEST_PARAMS["fee_rate_sell"],
#     lot_size=BACKTEST_PARAMS["lot_size"],
#     max_trade_amount_ratio=BACKTEST_PARAMS["max_trade_amount_ratio"],
#     amount_unit_scale=BACKTEST_PARAMS["amount_unit_scale"],
#     use_drift_trigger=BACKTEST_PARAMS["use_drift_trigger"],
#     drift_threshold=BACKTEST_PARAMS["drift_threshold"],
#     rp_prepare_kwargs=RP_PREPARE_KWARGS,
#     rp_weight_kwargs=RP_WEIGHT_KWARGS,
#     risk_free_rate=BACKTEST_PARAMS["risk_free_rate"],
#     annualization=BACKTEST_PARAMS["annualization"],
# )
